[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap09/cap09.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="19%"}](https://github.com/fzampirolli/pdi-vc)


## 💻 **Parte Prática com Exercícios de Programação**

🚧 **Em construção!**

A presente lista de exercícios de programação (EP) consolida as formulações teóricas apresentadas ao longo do Capítulo 7 — Classificação de Imagens e Reconhecimento de Padrões — por meio de uma trilha prática aplicada. Diferentemente da manipulação direta de pixels dos capítulos anteriores, os EPs deste capítulo trabalham com as **grandezas intermediárias** de um *pipeline* real de reconhecimento de padrões — vetores de características, distâncias, rótulos previstos e reais, códigos binários locais e histogramas de orientação — permitindo validar manualmente cada etapa do raciocínio sem depender de bibliotecas externas de aprendizado de máquina.

O encadeamento dos exercícios reproduz o fluxo conceitual do capítulo: inicia-se com a implementação manual da regra de decisão do classificador **k-NN** sobre um pequeno espaço de características; avança-se para o cálculo das métricas de **avaliação** (matriz de confusão, precisão e revocação) a partir de rótulos previstos e reais; prossegue-se com a codificação manual do descritor de textura **LBP** a partir de uma vizinhança $3\times3$; aprofunda-se no cálculo do histograma de orientações do descritor **HOG** para uma única célula; e conclui-se com a integração de **extração de descritores**, **classificação k-NN** e **avaliação multi-classe** em um *pipeline* completo de reconhecimento de texturas.

::: {.callout-important}
### Diretrizes para a Resolução dos Exercícios de Programação {.unnumbered}

Em todos os exercícios deste capítulo, as etapas de discretização ou arredondamento numérico devem empregar o arredondamento padrão para o inteiro mais próximo (*round half away from zero*), mitigando ambiguidades em valores com fração exatamente igual a $0{,}5$. Salvo indicação explícita em contrário, distâncias utilizam a métrica euclidiana, empates em votações são resolvidos pela regra descrita em cada exercício, e vetores/matrizes seguem indexação a partir de $0$, com a convenção `[linha][coluna]` para estruturas bidimensionais.
:::


### 🎯 Objetivo deste Caderno {.unnumbered}

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.


#### Download {.unnumbered}

Baixe `morph.py` e `testsuite.py` executando a célula abaixo:


In [ ]:
import os, sys, importlib, inspect, urllib.request

# URLs do repositório
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")


#### Executando os Testes {.unnumbered}
Para rodar os testes, execute `TestSuite("EP07_01.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

Para testar código Python diretamente, sem salvar arquivo, use `run_code(codigo)` passando o código como string numa variável `codigo`:

```python
codigo = """
# ... seu código aqui ...
"""
TestSuite("EP07_01").run_code(codigo)
```


### EP07_01 🟢 Classificador k-NN Passo a Passo

O `KNeighborsClassifier` do `scikit-learn`, usado ao longo do capítulo, esconde por trás de uma única chamada (`.fit` / `.predict`) uma regra de decisão bastante simples: para cada nova observação, calcular a distância a todos os exemplos de treinamento, selecionar os $k$ mais próximos e votar pela classe majoritária entre eles.

Antes de confiar na biblioteca, você foi encarregado de implementar essa regra do zero, para um espaço de características bidimensional, exatamente como o simulador interativo de fronteira de decisão do capítulo faz internamente a cada clique do usuário.

#### 📋 Diretrizes de Implementação

1. **Quantidade e parâmetro:** Ler o inteiro $N$ (número de exemplos de treinamento) e o inteiro ímpar $k$ (número de vizinhos).
2. **Exemplos de treinamento:** Para cada um dos $N$ exemplos, ler três valores: as coordenadas $x$ e $y$ (reais) e o rótulo $r$ (inteiro, $0$ ou $1$).
3. **Consultas:** Ler o inteiro $Q$ (número de pontos de consulta) e, em seguida, as coordenadas $x_q$, $y_q$ (reais) de cada consulta.
4. **Distância:** Para cada consulta, calcular a distância euclidiana até **todos** os exemplos de treinamento:
$$
d(x_q, x_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}.
$$
5. **Seleção dos vizinhos:** Ordenar os exemplos por distância crescente e selecionar os $k$ primeiros. Em caso de **empate de distância** na fronteira do k-ésimo vizinho, desempate pelo exemplo lido **primeiro** na entrada (ordem de leitura estável).
6. **Votação majoritária:** Contar os votos de cada classe entre os $k$ vizinhos selecionados. Se houver **empate na votação** (apenas possível quando $k$ é par, o que não deve ocorrer pela diretriz do item 1, mas trate defensivamente), atribua a classe do vizinho mais próximo entre as classes empatadas.
7. **Saída:** Para cada consulta, na ordem de entrada, imprimir a classe prevista. Ao final, imprimir o total de consultas classificadas como classe `1`.

#### 📌 Restrições Computacionais

* **Métrica fixa:** utilize exclusivamente a distância euclidiana (não a squared distance) para a ordenação, embora o resultado da comparação seja o mesmo.
* **k sempre ímpar:** a entrada garante $k$ ímpar e $k \le N$; ainda assim, implemente o desempate do item 6 por robustez.
* **Estabilidade:** ao ordenar por distância, preserve a ordem relativa de exemplos com a mesma distância (ordenação estável).

#### 🧠 Fundamentação Teórica

| Elemento | Papel no k-NN |
|---|---|
| Espaço de características | Conjunto de todos os vetores $(x, y)$ possíveis |
| Distância euclidiana | Medida de similaridade entre observações |
| $k$ pequeno | Fronteira irregular, alta variância |
| $k$ grande | Fronteira suave, alto viés |
| Votação majoritária | Regra de decisão $\hat y = \operatorname{moda}\{y_i : x_i \in N_k(x)\}$ |

#### 📦 Especificação de Entrada e Saída (VPL)

**Entrada:**

* Linha 1: Inteiros $N$ e $k$, separados por espaço.
* Próximas $N$ linhas: três valores por linha — $x$, $y$ (reais) e $r$ (inteiro $\in \{0,1\}$), separados por espaço.
* Próxima linha: inteiro $Q$.
* Próximas $Q$ linhas: dois valores por linha — $x_q$, $y_q$ (reais), separados por espaço.

**Saída:**

* $Q$ linhas, cada uma com a classe prevista (`0` ou `1`) para a respectiva consulta, na ordem de entrada.
* Última linha: `Total classe 1: X`.

#### 📌 Exemplos

| Entrada | Saída | Observação |
|---|---|---|
| 4 3<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>1<br>1 1 | 0<br>Total classe 1: 0 | Consulta próxima do agrupamento de classe 0. |
| 4 1<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>2<br>0.9 0.1<br>5.5 5.1 | 0<br>1<br>Total classe 1: 1 | Com $k=1$, cada consulta herda a classe do vizinho mais próximo. |


In [ ]:
#| label: fig-07-sim-ep01
#| fig-cap: "Simulador: Classificador k-NN Passo a Passo"
#| echo: false
#| output: true
from IPython.display import HTML
HTML(\'\'\'
<div id="sim-ep0701" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">
  <div style="background:#f3efe6;padding:8px 16px;font-size:12px;color:#5e5a4a;border-bottom:1px solid #e9dfcf;display:flex;justify-content:space-between;align-items:center;">
    <span>🎮 Simulador: Classificador k-NN Passo a Passo</span>
    <span style="background:#e8e0cf;border-radius:40px;padding:2px 10px;font-weight:600;font-size:10px;">🟢 votação majoritária</span>
  </div>
  <div style="padding:20px;background:white;">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">Ajuste k e veja quais exemplos de treinamento (ordenados por distância) participam da votação para a consulta fixa (★).</p>
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:20px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">k (número de vizinhos)</label>
        <span id="ep0701_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">3</span>
      </div>
      <input id="ep0701_sl" style="width:100%;accent-color:#2980b9;" max="7" min="1" step="2" type="range" value="3">
    </div>
    <div id="ep0701_cards" style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;"></div>
    <div id="ep0701_debug" style="margin-top:20px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";
    var query = {x:1, y:1};
    var pontos = [
      {nome:"A", x:0, y:0, r:0},
      {nome:"B", x:1, y:0, r:0},
      {nome:"C", x:5, y:5, r:1},
      {nome:"D", x:6, y:5, r:1},
      {nome:"E", x:2, y:2, r:0},
      {nome:"F", x:4, y:4, r:1},
      {nome:"G", x:0, y:2, r:0},
      {nome:"H", x:6, y:3, r:1}
    ];
    pontos.forEach(function(p){ p.d = Math.sqrt((p.x-query.x)**2 + (p.y-query.y)**2); });
    pontos.sort(function(a,b){ return a.d - b.d; });

    var slEl = root.querySelector(\'#ep0701_sl\');
    var vlEl = root.querySelector(\'#ep0701_vl\');
    var cards = root.querySelector(\'#ep0701_cards\');
    var dbg = root.querySelector(\'#ep0701_debug\');

    function render(){
      var k = parseInt(slEl.value);
      vlEl.textContent = k;
      cards.innerHTML = \'\';
      var votos = [0, 0];
      pontos.forEach(function(p, i){
        var dentro = i < k;
        if(dentro) votos[p.r]++;
        var div = document.createElement(\'div\');
        div.style.cssText = \'text-align:center;border-radius:10px;padding:10px 6px;font-size:11px;\' +
          (dentro ? (p.r===0 ? \'background:#dbeafe;border:1px solid #93c5fd;color:#1e3a8a;\' : \'background:#fee2e2;border:1px solid #fca5a5;color:#991b1b;\') : \'background:#f3f4f6;border:1px solid #e5e7eb;color:#9ca3af;\');
        div.innerHTML = \'<div style="font-weight:700;">\'+p.nome+\' (r=\'+p.r+\')</div>\' +
          \'<div style="font-family:monospace;margin:4px 0;">d=\'+p.d.toFixed(2)+\'</div>\' +
          \'<div style="font-weight:700;">\'+(dentro?\'VOTA\':\'-\')+\'</div>\';
        cards.appendChild(div);
      });
      var previsto = votos[1] > votos[0] ? 1 : 0;
      dbg.textContent = \'k=\' + k + \'  |  votos classe0=\' + votos[0] + \', classe1=\' + votos[1] + \'  |  classe prevista=\' + previsto;
    }
    slEl.addEventListener(\'input\', render);
    render();
  }
  function tryInit(){
    var root = document.getElementById(\'sim-ep0701\');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
\'\'\')


In [ ]:
%%writefile EP07_01.py
# Código Python


In [ ]:
TestSuite("EP07_01.py").run()
